<a href="https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/00_neural_networks_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for Maritime Vision Applications
## Session 0 · Part 1 — Artificial Neural Networks
*Torben Globisch · Visual and Analytic Computing in Ocean Technologies (VACOT)*

---

| Notebook | Topic |
|---|---|
| **→ 00** | **Artificial Neural Networks: neurons, layers, learning** |
| 01 | Computer Vision & an MLP on MNIST |
| 02 | Convolutional Neural Networks |
| 03 | Vision Foundation Models: the CNN era |
| 04 | Vision Transformers |
| 05 | Self-supervised pretraining & transfer |

**The story of this notebook**, told with one small dataset of iris flowers:
1. a single **neuron** draws one straight line, and you can find a good line by hand (🕹️ playground),
2. some problems need more than one line: **hidden layers + non-linear activations**,
3. how a network finds its weights by itself: **loss → gradient descent → backpropagation, layer by layer**,
4. the **training loop**, watched step by step on mini-batches of 5 flowers.

> 💡 **How to use this notebook:** run the cells top to bottom (`Shift + Enter`), or *Runtime → Run all* before the lecture.
> This is a show-and-tell notebook: there are no exercises. Figure and animation code is hidden in collapsed cells (🎨 / 🎞️ / 🕹️).
> Animations come with a player: pause them and step through them frame by frame with ◀ ▶.

In [ ]:
# @title ⚙️ Setup: imports, colours & helpers — run me first { display-mode: "form" }
import math, io, re, base64, time, urllib.request
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, FancyArrowPatch
from IPython.display import HTML, display
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

# Colour palette — the same colours as the lecture slides
NAVY, BLUE, LAVENDER, ORANGE, GREEN, RED, GREY, LIGHT = (
    "#2D3A8C", "#5B67B8", "#A3A9DB", "#E8702A", "#3E9E64", "#C0392B", "#8A8FA8", "#EEF0F7")
plt.rcParams.update({
    "figure.dpi": 100, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 11, "axes.titlecolor": NAVY,
    "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, GREEN, BLUE, RED, GREY]),
    "animation.html": "jshtml", "animation.embed_limit": 80,
})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)
print(f"PyTorch {torch.__version__} | device: {device}")
if device.type == "cpu":
    print("⚠️  No GPU found. In Colab: Runtime → Change runtime type → T4 GPU")

def play(anim):
    """Show a matplotlib animation as an HTML player (play / pause / step frame by frame)."""
    plt.close(anim._fig)
    def shrink(m):  # re-encode every frame as a 256-colour PNG: 3-4x smaller, looks the same
        img = Image.open(io.BytesIO(base64.b64decode(m.group(1).replace("\\\n", ""))))
        buf = io.BytesIO(); img.convert("RGB").quantize(256).save(buf, "png", optimize=True)
        return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
    return HTML(re.sub(r"data:image/png;base64,([A-Za-z0-9+/=\\\n]+)", shrink, anim.to_jshtml()))

# Maritime example photos from the COCO validation set (Flickr, Creative Commons licences).
MARITIME = {"trawler": 330369, "sailboat": 286422, "lifeboat": 400803, "marina": 442822,
            "fishing_boat": 451090, "yacht": 489014, "ferry": 543300}

def load_image(name, size=None):
    """Load one of the MARITIME photos (or any image URL) as an RGB PIL image."""
    url = f"http://images.cocodataset.org/val2017/{MARITIME[name]:012d}.jpg" if name in MARITIME else name
    with urllib.request.urlopen(url, timeout=30) as r:
        img = Image.open(io.BytesIO(r.read())).convert("RGB")
    if size is not None:  # centre-crop to a square, then resize
        s = min(img.size); l, t = (img.width - s) // 2, (img.height - s) // 2
        img = img.crop((l, t, l + s, t + s)).resize((size, size), Image.BICUBIC)
    return img

---
# 1 · What is an Artificial Neural Network?

**Inspired by the brain.** The human brain has about 86 billion neurons connected by trillions of synapses.
A neuron collects signals through its *dendrites*, integrates them in the *soma* and, if the total
is strong enough, *fires* a signal down its *axon*.

An artificial neural network (ANN) copies this structure in a very simplified way:

| Biology | ANN |
|---|---|
| neuron | mathematical node |
| synapse strength | learnable **weight** $w$ |
| firing | **activation function** $\sigma$ |

> **Definition.** An ANN is a *parameterised function* $f_\theta : \mathbb{R}^n \to \mathbb{R}^m$, built from
> layers of neurons, that transforms an input into an output. *Learning* means finding good parameters $\theta$.

## 1.1 · The artificial neuron (Rosenblatt's perceptron, 1958)

Each neuron computes a **weighted sum** of its inputs, adds a **bias**, and applies an **activation function**:

$$ y = \sigma\Big(\sum_{i=1}^{n} w_i x_i + b\Big) = \sigma(\mathbf{w}^\top \mathbf{x} + b) $$

* $x_i$: input features   * $w_i$: weights (*learned*)   * $b$: bias (*learned*)   * $\sigma$: activation function

In [ ]:
# @title 🎨 Figure: the artificial neuron { display-mode: "form" }
def draw_neuron(ax):
    ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")
    ys = [4.8, 3.0, 1.2]
    for i, y in enumerate(ys, 1):
        ax.add_patch(Circle((1, y), 0.45, fc=LAVENDER, ec=NAVY, lw=1.5))
        ax.text(1, y, f"$x_{i}$", ha="center", va="center", fontsize=13)
        ax.annotate("", xy=(4.45, 3.0), xytext=(1.45, y), arrowprops=dict(arrowstyle="-|>", color=NAVY, lw=1.4))
        ax.text(2.9, (y + 3.0) / 2 + 0.25, f"$w_{i}$", color=ORANGE, fontsize=12, ha="center")
    ax.add_patch(Circle((5, 3), 0.6, fc="white", ec=NAVY, lw=2)); ax.text(5, 3, r"$\Sigma$", ha="center", va="center", fontsize=16)
    ax.annotate("", xy=(5, 2.4), xytext=(5, 0.9), arrowprops=dict(arrowstyle="-|>", color=GREEN, lw=1.4))
    ax.text(5, 0.6, "$b$", color=GREEN, ha="center", fontsize=13)
    ax.add_patch(FancyBboxPatch((6.4, 2.45), 1.1, 1.1, boxstyle="round,pad=0.05", fc=ORANGE, ec=ORANGE, alpha=.9))
    ax.text(6.95, 3, r"$\sigma$", ha="center", va="center", fontsize=16, color="white")
    ax.annotate("", xy=(6.35, 3), xytext=(5.6, 3), arrowprops=dict(arrowstyle="-|>", color=NAVY, lw=1.4))
    ax.annotate("", xy=(9.0, 3), xytext=(7.6, 3), arrowprops=dict(arrowstyle="-|>", color=NAVY, lw=1.4))
    ax.text(9.3, 3, "$y$", fontsize=15, va="center")
    ax.text(5, 5.6, r"$y = \sigma(w_1x_1 + w_2x_2 + w_3x_3 + b)$", ha="center", fontsize=13, color=NAVY)

fig, ax = plt.subplots(figsize=(7, 4)); draw_neuron(ax); plt.show()

In code, a neuron is one line. Let's compute one by hand:

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])     # inputs
w = torch.tensor([0.5, -1.0, 0.25])   # weights
b = torch.tensor(0.1)                 # bias

z = w @ x + b                         # weighted sum: 0.5·1 − 1·2 + 0.25·3 + 0.1
y = torch.relu(z)                     # activation
print(f"z = {z.item():.2f}   →   y = relu(z) = {y.item():.2f}")

## 1.2 · Our running example: iris petals 🌸

The *Iris* dataset (Fisher, 1936) contains 150 flowers of three species. For each flower we use two measurements:
**petal length** and **petal width**. Our question throughout this notebook is:

> **Is this flower a *versicolor*?**  (1 = yes, 0 = no)

We start with the easier half of the problem, **Task A**: tell *versicolor* apart from *virginica*.
The neuron gets the two measurements as inputs $x_1, x_2$ (standardised: 0 = an average flower, ±1 = one standard
deviation away) and uses a **sigmoid** $\sigma(z) = \frac{1}{1+e^{-z}}$ as its activation, so its output is a probability:

$$ p(\text{versicolor}) = \sigma(w_1 x_1 + w_2 x_2 + b) $$

In [ ]:
# @title 📷 What does a *versicolor* actually look like? { display-mode: "form" }
url = "https://upload.wikimedia.org/wikipedia/commons/4/41/Iris_versicolor_3.jpg"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})   # Wikimedia blocks urllib's default UA
with urllib.request.urlopen(req, timeout=30) as r:
    photo = Image.open(io.BytesIO(r.read())).convert("RGB")

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.imshow(photo); ax.axis("off")
ax.set_title("Iris versicolor", color=NAVY, fontweight="bold")
plt.tight_layout(); plt.show()
print("Photo: Danielle Langlois, CC BY-SA 3.0, via Wikimedia Commons.")

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
SPECIES = list(iris.target_names)             # ['setosa', 'versicolor', 'virginica']
petals_cm = iris.data[:, 2:4]                 # 150 flowers × (petal length, petal width) in cm
species = iris.target                         # 0 = setosa, 1 = versicolor, 2 = virginica

# The neuron gets standardised inputs: 0 = an average flower, ±1 = one standard deviation away
X_all = ((petals_cm - petals_cm.mean(0)) / petals_cm.std(0)).astype(np.float32)
y_all = (species == 1).astype(np.float32)     # our question: "is it a versicolor?"  (1 = yes, 0 = no)
print("flowers per species:", dict(zip(SPECIES, np.bincount(species).tolist())))

In [ ]:
# @title 🎨 Plot helpers for the iris flowers { display-mode: "form" }
import warnings
from matplotlib.colors import LinearSegmentedColormap

SP_COLORS, SP_MARKERS = [GREEN, ORANGE, NAVY], ["s", "o", "^"]          # setosa, versicolor, virginica
P_CMAP = LinearSegmentedColormap.from_list("p", [BLUE, "white", ORANGE])   # p(versicolor): 0 → blue, 1 → orange
H_CMAP = LinearSegmentedColormap.from_list("h", ["#7B4FA0", "white", GREEN])
EXT = (-2.0, 2.2)                                                          # plotting range (standardised units)
G1, G2 = np.meshgrid(np.linspace(*EXT, 120), np.linspace(*EXT, 120))
GRID = np.stack([G1.ravel(), G2.ravel()], 1).astype(np.float32)

def plot_flowers(ax, mask=None, s=28, legend=False, labels=True):
    """Scatter the flowers (standardised petal length/width), coloured by species."""
    mask = np.ones(len(species), bool) if mask is None else mask
    for k in range(3):
        m = mask & (species == k)
        if m.any():
            ax.scatter(*X_all[m].T, s=s, c=SP_COLORS[k], marker=SP_MARKERS[k], ec=(0, 0, 0, .55), lw=.5,
                       label=SPECIES[k], zorder=3)
    ax.set(xlim=EXT, ylim=EXT); ax.set_aspect("equal")
    if labels: ax.set(xlabel="petal length $x_1$", ylabel="petal width $x_2$")
    else: ax.set_xticks([]); ax.set_yticks([])
    if legend: ax.legend(loc="upper left", fontsize=8, framealpha=.9)

def contour_at(ax, Z, level, **kw):
    """Draw one contour line of a grid map, but only if the map actually crosses that level."""
    Z = Z.reshape(G1.shape)
    if Z.min() < level < Z.max():
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return ax.contour(G1, G2, Z, levels=[level], **kw)
    return None

def plot_prob(ax, p_grid, line=True, **kw):
    """Background = the model's p(versicolor) everywhere; the 0.5 contour is its decision boundary."""
    im = ax.imshow(p_grid.reshape(G1.shape), extent=(*EXT, *EXT), origin="lower", cmap=P_CMAP,
                   vmin=0, vmax=1, alpha=.55, zorder=0)
    cs = contour_at(ax, p_grid, 0.5, colors=NAVY, linewidths=2.2, zorder=2, **kw) if line else None
    return im, cs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
for k in range(3):
    axes[0].scatter(*petals_cm[species == k].T, s=28, c=SP_COLORS[k], marker=SP_MARKERS[k], ec="white", lw=.6, label=SPECIES[k])
axes[0].set(xlabel="petal length (cm)", ylabel="petal width (cm)", title="All 150 flowers"); axes[0].legend()
plot_flowers(axes[1], mask=species > 0, legend=True)
axes[1].set_title("Task A: versicolor or virginica?\n(standardised inputs)")
plt.tight_layout(); plt.show()

### 🕹️ Play with a neuron
Three sliders, three parameters: $w_1$, $w_2$ and $b$. Try to make the neuron answer Task A correctly.

* **Right: the solution.** The background colour is the neuron's answer $p(\text{versicolor})$ at every point; the navy line
  ($w_1x_1 + w_2x_2 + b = 0$, i.e. $p = 0.5$) is its **decision boundary**. Weights **rotate** the line, the bias **shifts** it.
  Flowers on the wrong side are circled in red.
* **Left: the loss.** One number that says how wrong the neuron is:
  the average over all flowers of $-\log(\text{probability the neuron gives to the correct answer})$.
  The map shows the loss for *every* $(w_1, w_2)$ at the current $b$; the orange dot is you, the trail is where you have been.
  Dark = low loss = good. **Learning means walking into the dark valley.**

Then switch the dropdown to **Task B** (find the versicolor among *all three* species) and try again.

In [ ]:
# @title 🕹️ Neuron playground (run me, then move the sliders) { display-mode: "form" }
from ipywidgets import interact, FloatSlider, Dropdown

TASKS = {"A: versicolor vs. virginica": species > 0,          # which flowers take part in the task
         "B: versicolor vs. setosa + virginica": species >= 0}

def neuron(X, w1, w2, b):
    """One neuron: weighted sum + bias, then a sigmoid → probability of 'versicolor'."""
    return 1 / (1 + np.exp(-(w1 * X[..., 0] + w2 * X[..., 1] + b)))

def bce(p, y):
    """Binary cross-entropy: average of −log(probability given to the correct answer)."""
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean(-1)

W_RANGE = 8
WG1, WG2 = np.meshgrid(np.linspace(-W_RANGE, W_RANGE, 81), np.linspace(-W_RANGE, W_RANGE, 81))
LOSS_LEVELS = [0.15, 0.2, 0.3, 0.45, 0.7, 1, 2, 4]
trail = {"task": None, "pts": []}

def playground(task, w1, w2, b):
    m = TASKS[task]; X, y = X_all[m], y_all[m]
    if trail["task"] != task: trail.update(task=task, pts=[])
    trail["pts"].append((w1, w2))
    landscape = bce(neuron(X[None, None], WG1[..., None], WG2[..., None], b), y)   # loss for every (w1, w2)
    p = neuron(X, w1, w2, b); loss = bce(p, y); wrong = (p > 0.5) != y

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(12.5, 5.2))
    im = axL.imshow(landscape, extent=(-W_RANGE, W_RANGE, -W_RANGE, W_RANGE), origin="lower", cmap="Blues_r",
                    norm=mpl.colors.LogNorm(vmin=0.1, vmax=8))
    axL.contour(WG1, WG2, landscape, levels=LOSS_LEVELS, colors="white", linewidths=.6, alpha=.8)
    path = np.array(trail["pts"][-60:])
    axL.plot(*path.T, "-", color=ORANGE, lw=1.3, alpha=.8)
    axL.plot(w1, w2, "o", ms=12, color=ORANGE, mec="white", mew=2)
    axL.set(xlabel="$w_1$", ylabel="$w_2$", title=f"LOSS = {loss:.3f}\n(landscape over $w_1, w_2$ at b = {b:.1f})")
    plt.colorbar(im, ax=axL, label="loss  (lower = better)", shrink=.85)

    plot_prob(axR, neuron(GRID, w1, w2, b))
    plot_flowers(axR, mask=m, legend=True)
    axR.scatter(*X[wrong].T, s=120, facecolors="none", edgecolors=RED, lw=1.8, zorder=4)
    axR.set_title(f"p = σ({w1:.1f}·x₁ {w2:+.1f}·x₂ {b:+.1f})\naccuracy {1 - wrong.mean():.0%}  "
                  f"({(~wrong).sum()} of {len(y)} correct)")
    plt.tight_layout(); plt.show()

slider = lambda v: FloatSlider(value=v, min=-W_RANGE, max=W_RANGE, step=0.1)
interact(playground, task=Dropdown(options=list(TASKS), description="task"),
         w1=slider(1.0), w2=slider(-1.0), b=slider(0.0));

**What did we see?**

* **Task A** can be solved *okayish* by hand: around 93 % is possible. A few flowers always stay on the wrong side because the
  two species overlap; no straight line gets them all. Notice how lower loss on the left goes together with a better line on the right.
* **Task B** cannot be solved by one neuron. *Versicolor* sits **between** setosa (small petals) and virginica (large petals),
  and a single neuron can only cut the plane into two halves with one straight line. Whatever you do, one of the other species
  ends up on the versicolor side. The best compromise is about 80 %, and the loss never gets really low.

**So we need more than one line.** Let's build a small neural network for Task B.

---
# 2 · From one neuron to a network: why do we need non-linearity?

**Idea:** use two neurons, each drawing its own line, and a third neuron that combines their answers:

* hidden neuron 1: *"are the petals bigger than a setosa's?"*
* hidden neuron 2: *"are the petals bigger than a versicolor's?"*
* output neuron: *"yes to neuron 1 **and** no to neuron 2"* → versicolor!

The two neurons in the middle form a **hidden layer**. Together this is a tiny artificial neural network, **2 → 2 → 1**, with
$2\cdot2+2 = 6$ parameters in the hidden layer and $2\cdot1+1 = 3$ in the output layer: **9 parameters**.

In [ ]:
# @title 🎨 Figure helper: draw a fully-connected network { display-mode: "form" }
def draw_mlp(ax, sizes, names=None, max_show=8):
    """Draw a dense network. Layers wider than `max_show` are drawn truncated with '⋮'."""
    ax.axis("off"); L = len(sizes); m = min(max(sizes), max_show)
    colors = [BLUE] + [LAVENDER] * (L - 2) + [ORANGE]
    pos = []
    for l, n in enumerate(sizes):
        shown = min(n, max_show)
        ys = np.linspace(-(shown - 1) / 2, (shown - 1) / 2, shown) if shown > 1 else np.array([0.0])
        pos.append([(l * 3.0, y) for y in ys])
    for l in range(L - 1):
        for (x0, y0) in pos[l]:
            for (x1, y1) in pos[l + 1]:
                ax.plot([x0, x1], [y0, y1], color=GREY, lw=0.5, alpha=0.6, zorder=1)
    for l, layer in enumerate(pos):
        for (x, y) in layer:
            ax.add_patch(Circle((x, y), 0.32, fc=colors[l], ec=NAVY, lw=1.2, zorder=2))
        if sizes[l] > max_show:
            ax.text(layer[0][0], 0, "⋮", ha="center", va="center", fontsize=16, zorder=3, color=NAVY,
                    bbox=dict(fc="white", ec="none"))
        label = (names[l] if names else ("Input" if l == 0 else "Output" if l == L - 1 else f"Hidden {l}"))
        ax.text(layer[0][0], -m / 2 - 0.3, f"{label}\nn = {sizes[l]}", ha="center", va="top", fontsize=10, color=NAVY)
    ax.set_xlim(-1, (L - 1) * 3 + 1); ax.set_ylim(-m / 2 - 1.8, m / 2 + 0.2); ax.set_aspect("equal")

fig, ax = plt.subplots(figsize=(5, 2.6))
draw_mlp(ax, [2, 2, 1], ["Input\n(length, width)", "Hidden\n(2 neurons)", "Output\np(versicolor)"]); plt.show()

Let's pick the 9 weights **by hand** and look at what every neuron computes. The hidden neurons use the activation
$\tanh$ (an S-curve from −1 to +1, like the sigmoid but centred at 0).
Top row: **with** the activation. Bottom row: the **same weights without** the activation.

In [ ]:
# @title 🎨 Figure helper: what does every neuron of a 2 → 2 → 1 network compute? { display-mode: "form" }
def network_maps(axes, W1, b1, w2, b2, act=np.tanh, act_name="tanh"):
    """Plot hidden neuron 1, hidden neuron 2 and the output p(versicolor) over the whole input plane."""
    Z1 = GRID @ W1.T + b1                                   # hidden pre-activations on the grid
    H = act(Z1)                                             # hidden activations
    p = 1 / (1 + np.exp(-(H @ w2 + b2)))                    # output neuron (sigmoid)
    lim = 1 if act_name == "tanh" else 4
    for j in range(2):
        ax = axes[j]
        ax.imshow(H[:, j].reshape(G1.shape), extent=(*EXT, *EXT), origin="lower", cmap=H_CMAP, vmin=-lim, vmax=lim, alpha=.6)
        contour_at(ax, Z1[:, j], 0, colors=NAVY, linewidths=1.5, linestyles="--")
        plot_flowers(ax, s=14, labels=False)
        ax.set_title(f"hidden neuron {j + 1}\n$h_{j + 1}$ = {act_name}({W1[j, 0]:g}·x₁ {W1[j, 1]:+g}·x₂ {b1[j]:+g})", fontsize=10)
    H_data = act(X_all @ W1.T + b1)
    acc = (((H_data @ w2 + b2) > 0) == y_all).mean()
    plot_prob(axes[2], p); plot_flowers(axes[2], s=14, labels=False)
    axes[2].set_title(f"output: p = σ({w2[0]:g}·h₁ {w2[1]:+g}·h₂ {b2:+g})\naccuracy on all 150 flowers: {acc:.0%}", fontsize=10)
    return acc

In [ ]:
# Hand-picked weights for the 2 → 2 → 1 network
W1 = np.array([[3.0, 3.0],      # hidden neuron 1: "bigger than a setosa?"
               [1.0, 1.0]])     # hidden neuron 2: "bigger than a versicolor?"
b1 = np.array([3.9, -1.25])
w2 = np.array([3.0, -3.0])      # output neuron:   "yes to neuron 1 AND no to neuron 2"
b2 = -3.0

fig, axes = plt.subplots(2, 3, figsize=(12, 8.4))
network_maps(axes[0], W1, b1, w2, b2, np.tanh, "tanh")
network_maps(axes[1], W1, b1, w2, b2, lambda z: z, "identity")
axes[0, 0].set_ylabel("WITH activation (tanh)", fontsize=12, color=NAVY, fontweight="bold")
axes[1, 0].set_ylabel("WITHOUT activation", fontsize=12, color=RED, fontweight="bold")
plt.tight_layout(); plt.show()

* **With tanh**, each hidden neuron gives a clean *yes* (+1, green) or *no* (−1, purple) on its side of its line, and the output
  neuron combines the two answers into a **band**: 96 % correct, far better than any single line.
* **Without an activation**, each hidden "neuron" is just a tilted plane, and adding tilted planes gives another tilted plane.
  The output collapses back into **one straight line**: we are back to a single neuron.

This is true in general: two linear layers $W_2(W_1x + b_1) + b_2$ are just **one** linear layer
$Wx + b$ with $W = W_2W_1$ and $b = W_2b_1 + b_2$. However deep you stack them, without activations you only ever get a straight
boundary. A quick numerical check with three layers:

In [ ]:
torch.manual_seed(0)
deep_linear = nn.Sequential(nn.Linear(2, 16), nn.Linear(16, 16), nn.Linear(16, 2))   # 3 layers, no activation

# Collapse the three layers into a single (W, b)
W = torch.eye(2); b = torch.zeros(2)
for layer in deep_linear:
    W, b = layer.weight @ W, layer.weight @ b + layer.bias

x = torch.randn(5, 2)
print("3-layer network output:\n", deep_linear(x).detach())
print("single linear map  W x + b:\n", (x @ W.T + b).detach())
print("identical:", torch.allclose(deep_linear(x), x @ W.T + b, atol=1e-6))

## 2.1 · Activation functions

| Name | Formula | Notes |
|---|---|---|
| **ReLU** | $\max(0, x)$ | default choice in modern (deep) networks: cheap, does not saturate for $x>0$ |
| **Sigmoid** | $\frac{1}{1+e^{-x}}$ | squashes to $(0,1)$, used for probabilities (our output neuron) |
| **Tanh** | $\tanh(x)$ | squashes to $(-1,1)$, zero-centred (our hidden neurons) |

The **derivative** matters as much as the function itself: it decides how much gradient flows back during training (Section 3).

In [ ]:
xs = torch.linspace(-5, 5, 400, requires_grad=True)
fig, axes = plt.subplots(2, 3, figsize=(11, 5), sharex=True)
for col, (name, fn, color) in enumerate([("ReLU", torch.relu, NAVY), ("Sigmoid", torch.sigmoid, ORANGE), ("Tanh", torch.tanh, GREEN)]):
    ys = fn(xs)
    grad, = torch.autograd.grad(ys.sum(), xs)          # derivative via autograd
    axes[0, col].plot(xs.detach(), ys.detach(), color=color, lw=2.5); axes[0, col].set_title(name)
    axes[1, col].plot(xs.detach(), grad, color=color, lw=2.5, ls="--")
    for ax in axes[:, col]: ax.axhline(0, color=GREY, lw=.6); ax.axvline(0, color=GREY, lw=.6)
axes[0, 0].set_ylabel("σ(x)"); axes[1, 0].set_ylabel("σ'(x)  (derivative)")
for ax in axes[1]: ax.set_xlabel("x")
plt.tight_layout(); plt.show()

## 2.2 · Layers and parameters

Real networks have the same structure, just bigger:
* **Input layer**: the raw data (pixel values, sensor readings, petal sizes, …)
* **Hidden layers**: learn intermediate representations (like our two "bigger than …?" neurons)
* **Output layer**: the final prediction (e.g. one score per class)

In a **fully-connected (dense) layer** every neuron is connected to every neuron of the next layer.
A dense layer with $n_\text{in}$ inputs and $n_\text{out}$ outputs has $n_\text{in}\cdot n_\text{out}$ weights **plus** $n_\text{out}$ biases.
**Hyperparameters** are chosen by *you*, not learned: depth (# layers) · width (neurons per layer) · activation function · learning rate $\eta$.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2)); draw_mlp(ax, [3, 4, 4, 2]); plt.show()

model = nn.Sequential(
    nn.Linear(3, 4), nn.ReLU(),   # input  → hidden 1
    nn.Linear(4, 4), nn.ReLU(),   # hidden 1 → hidden 2
    nn.Linear(4, 2),              # hidden 2 → output
)
print(model)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\nparameters per layer:", [(n, p.numel()) for n, p in model.named_parameters()])
print("total:", count_params(model), "= (3·4+4) + (4·4+4) + (4·2+2)")

Picking the 9 weights of our iris network by hand already took some thinking. The MLP in the next notebook has 235,146
parameters, modern vision models have millions to billions. **Nobody picks those by hand: the network has to learn them.**

---
# 3 · How does an ANN learn?

Three steps, repeated over and over:

1. **Forward pass**: the input $x$ flows layer by layer to a prediction $\hat y$.
2. **Loss**: one number that measures how wrong the prediction is. You already met it: the left panel of the playground.
3. **Backpropagation + update**: compute $\frac{\partial \mathcal{L}}{\partial w}$ for **every** weight (which way is downhill?), then
   take a small step downhill: $\;w \leftarrow w - \eta \, \frac{\partial \mathcal{L}}{\partial w}$

## 3.1 · The loss
For a yes/no question we use the **binary cross-entropy**, the average over all $N$ examples of the
"surprise" about the correct answer:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} \Big[\, y_i \log p_i + (1-y_i)\log(1-p_i) \Big]$$

For a versicolor ($y=1$) only $-\log p$ counts, for any other flower only $-\log(1-p)$. Confident mistakes are punished very hard:

In [ ]:
p = np.linspace(0.001, 1, 400)
fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.plot(p, -np.log(p), color=ORANGE, lw=2.5)
ax.set(xlabel="probability given to the correct answer", ylabel="loss  $-\\log p$", title="Cross-entropy for one example", ylim=(0, 6))
ax.annotate("confident & wrong\n→ huge loss", xy=(0.02, 3.9), xytext=(0.25, 4.6), arrowprops=dict(arrowstyle="->", color=NAVY))
ax.annotate("confident & right\n→ loss ≈ 0", xy=(0.97, 0.05), xytext=(0.6, 1.6), arrowprops=dict(arrowstyle="->", color=NAVY))
plt.show()

## 3.2 · Gradient descent: let the computer move the sliders
In the playground *you* moved $w_1, w_2, b$ and watched the loss. The **gradient** $\big(\frac{\partial \mathcal{L}}{\partial w_1},
\frac{\partial \mathcal{L}}{\partial w_2}, \frac{\partial \mathcal{L}}{\partial b}\big)$ tells us, for every slider, in which direction the
loss goes *up*, so we move each slider a small step in the *opposite* direction. PyTorch computes the gradient for us with
`loss.backward()`. Here is the whole algorithm for Task A, starting where the playground started:

In [ ]:
XA, yA = torch.tensor(X_all[species > 0]), torch.tensor(y_all[species > 0])   # Task A: 100 flowers

w = torch.tensor([1.0, -1.0], requires_grad=True)       # the playground's starting point
b = torch.tensor(0.0, requires_grad=True)
lr = 3.0                                                # learning rate η: the step size
gd_history = []
for step in range(41):
    p = torch.sigmoid(XA @ w + b)                       # 1. forward pass: the neuron's answer for all 100 flowers
    loss = F.binary_cross_entropy(p, yA)                # 2. loss
    loss.backward()                                     # 3. gradients ∂L/∂w₁, ∂L/∂w₂, ∂L/∂b
    gd_history.append(dict(w=w.tolist(), b=b.item(), grad=w.grad.tolist(), loss=loss.item(),
                           acc=((p > 0.5) == yA).float().mean().item()))
    with torch.no_grad():                               # 4. one step downhill
        w -= lr * w.grad;  b -= lr * b.grad
        w.grad.zero_();    b.grad.zero_()

h = gd_history[-1]
print(f"after {step} steps: w = ({h['w'][0]:.2f}, {h['w'][1]:.2f}), b = {h['b']:.2f}  →  loss {h['loss']:.3f}, accuracy {h['acc']:.0%}")

In [ ]:
# @title 🎞️ Animation: gradient descent on the neuron playground { display-mode: "form" }
XA_np, yA_np = X_all[species > 0], y_all[species > 0]
fig, (axL, axR, axC) = plt.subplots(1, 3, figsize=(15.5, 4.8), dpi=85, gridspec_kw=dict(width_ratios=[1.15, 1, 1]))

landscape_at = lambda b: bce(neuron(XA_np[None, None], WG1[..., None], WG2[..., None], b), yA_np)
im = axL.imshow(landscape_at(0.0), extent=(-W_RANGE, W_RANGE, -W_RANGE, W_RANGE), origin="lower", cmap="Blues_r",
                norm=mpl.colors.LogNorm(vmin=0.1, vmax=8))
plt.colorbar(im, ax=axL, label="loss", shrink=.85)
path_line, = axL.plot([], [], "-o", color=ORANGE, ms=3, lw=1.5, mfc="white")
dot, = axL.plot([], [], "o", ms=12, color=ORANGE, mec="white", mew=2)
arrow = [None]; level_lines = [None]
axL.set(xlabel="$w_1$", ylabel="$w_2$")

plot_flowers(axR, mask=species > 0, legend=True)
bg = axR.imshow(np.zeros(G1.shape), extent=(*EXT, *EXT), origin="lower", cmap=P_CMAP, vmin=0, vmax=1, alpha=.55, zorder=0)
boundary = [None]

steps = np.arange(len(gd_history))
L = np.array([h["loss"] for h in gd_history]); A = np.array([h["acc"] for h in gd_history])
axC.plot(steps, L, color=NAVY, lw=.8, alpha=.25); axC2 = axC.twinx()
loss_curve, = axC.plot([], [], color=NAVY, lw=2.5, label="loss")
acc_curve, = axC2.plot([], [], color=GREEN, lw=2.5, label="accuracy")
axC.set(xlabel="step", ylabel="loss", xlim=(0, steps[-1]), ylim=(0, L.max() * 1.1), title="Loss & accuracy")
axC2.set(ylim=(0.4, 1.0), ylabel="accuracy"); axC2.spines["right"].set_visible(True)
axC.legend(handles=[loss_curve, acc_curve], loc="center right")
plt.tight_layout()

def update(t):
    h = gd_history[t]; (w1, w2), b = h["w"], h["b"]
    Z = landscape_at(b); im.set_data(Z)
    if level_lines[0] is not None: level_lines[0].remove()
    level_lines[0] = axL.contour(WG1, WG2, Z, levels=LOSS_LEVELS, colors="white", linewidths=.6, alpha=.8)
    pts = np.array([g["w"] for g in gd_history[:t + 1]])
    path_line.set_data(*pts.T); dot.set_data([w1], [w2])
    if arrow[0] is not None: arrow[0].remove()
    g = -np.array(h["grad"]); g = g / (np.linalg.norm(g) + 1e-9) * 2.2       # direction of steepest descent
    arrow[0] = axL.annotate("", xy=(w1 + g[0], w2 + g[1]), xytext=(w1, w2),
                            arrowprops=dict(arrowstyle="-|>", color=RED, lw=2.2))
    axL.set_title(f"step {t}: loss = {h['loss']:.3f}\nred arrow = −gradient (downhill)")
    bg.set_data(neuron(GRID, w1, w2, b).reshape(G1.shape))
    if boundary[0] is not None: boundary[0].remove()
    boundary[0] = contour_at(axR, neuron(GRID, w1, w2, b), 0.5, colors=NAVY, linewidths=2.2, zorder=2)
    axR.set_title(f"p = σ({w1:.2f}·x₁ {w2:+.2f}·x₂ {b:+.2f})\naccuracy {h['acc']:.0%}")
    loss_curve.set_data(steps[:t + 1], L[:t + 1]); acc_curve.set_data(steps[:t + 1], A[:t + 1])
    return []

play(animation.FuncAnimation(fig, update, frames=len(gd_history), interval=220))

Gradient descent does exactly what you did by hand, only systematically: at every step it looks at the slope of the loss
landscape (red arrow) and walks downhill. The line on the right swings into place by itself.

**The learning rate $\eta$** sets the step size: here $\eta = 3$. We will see what a bad choice does at the end of the
training section.

## 3.3 · Backpropagation: the chain rule, one layer at a time
For one neuron the gradient is easy. How do we get $\frac{\partial \mathcal{L}}{\partial w}$ for a weight deep inside a network?
Take our 2 → 2 → 1 iris network. The forward pass is a chain of simple steps:

$$ \underbrace{z_1 = W_1x + b_1}_{\text{hidden layer}} \;\rightarrow\; h = \tanh(z_1) \;\rightarrow\;
\underbrace{z_2 = w_2^\top h + b_2}_{\text{output layer}} \;\rightarrow\; p = \sigma(z_2) \;\rightarrow\; \mathcal{L} = -\log p \;\; (\text{for } y = 1)$$

**Backpropagation** walks this chain **backwards**. Each layer receives the gradient from the layer above it, multiplies it
with its own *local* derivative (chain rule) and passes the result down to the layer below:

| backward step | formula |
|---|---|
| output layer: error signal | $\delta_2 = \frac{\partial \mathcal{L}}{\partial z_2} = p - y$ (sigmoid + cross-entropy simplify nicely) |
| → gradients of the output layer | $\frac{\partial \mathcal{L}}{\partial w_2} = \delta_2\, h, \qquad \frac{\partial \mathcal{L}}{\partial b_2} = \delta_2$ |
| hand the signal down | $\frac{\partial \mathcal{L}}{\partial h} = \delta_2\, w_2$ |
| through the activation | $\delta_1 = \frac{\partial \mathcal{L}}{\partial z_1} = \frac{\partial \mathcal{L}}{\partial h} \odot (1 - h^2) \qquad (\tanh' = 1-\tanh^2)$ |
| → gradients of the hidden layer | $\frac{\partial \mathcal{L}}{\partial W_1} = \delta_1\, x^\top, \qquad \frac{\partial \mathcal{L}}{\partial b_1} = \delta_1$ |

Every layer only needs *its own* inputs and the signal coming from above. That is why it scales to networks with hundreds of
layers. The animation runs one forward and one backward pass for a single flower, with real numbers:

In [ ]:
SEED = 3                                                     # the same starting weights are used for training below
torch.manual_seed(SEED)
net = nn.Sequential(nn.Linear(2, 2), nn.Tanh(), nn.Linear(2, 1))   # 2 → 2 → 1, random initial weights
FLOWER = 70                                                  # one versicolor flower
x0, y0 = torch.tensor(X_all[FLOWER]), torch.tensor(y_all[FLOWER])
print(f"flower #{FLOWER}: {SPECIES[species[FLOWER]]}, petal {petals_cm[FLOWER, 0]} × {petals_cm[FLOWER, 1]} cm"
      f"  →  x = ({x0[0]:.2f}, {x0[1]:.2f}),  y = {y0:.0f}")

In [ ]:
# @title 🎞️ Animation: forward & backward pass through the 2 → 2 → 1 network { display-mode: "form" }
# --- all numbers, computed by hand in numpy ---
W1n = net[0].weight.detach().numpy().astype(float); b1n = net[0].bias.detach().numpy().astype(float)
w2n = net[2].weight.detach().numpy()[0].astype(float); b2n = float(net[2].bias.detach().numpy()[0])
xn, yn, ETA = X_all[FLOWER].astype(float), float(y_all[FLOWER]), 1.0
z1 = W1n @ xn + b1n; hn = np.tanh(z1); z2 = w2n @ hn + b2n; pn = 1 / (1 + np.exp(-z2))
Ln = -(yn * np.log(pn) + (1 - yn) * np.log(1 - pn))
d_z2 = pn - yn; d_w2 = d_z2 * hn; d_b2 = d_z2; d_h = d_z2 * w2n; d_z1 = d_h * (1 - hn**2)
d_W1 = np.outer(d_z1, xn); d_b1 = d_z1
W1u, b1u, w2u, b2u = W1n - ETA * d_W1, b1n - ETA * d_b1, w2n - ETA * d_w2, b2n - ETA * d_b2
pu = 1 / (1 + np.exp(-(w2u @ np.tanh(W1u @ xn + b1u) + b2u))); Lu = -np.log(pu) if yn == 1 else -np.log(1 - pu)

# --- layout ---
POS = {"x0": (0, 2.6), "x1": (0, -0.6), "h0": (4, 2.6), "h1": (4, -0.6), "out": (8, 1.0), "L": (11, 1.0)}
fig, ax = plt.subplots(figsize=(13.5, 7.4)); ax.set_xlim(-1.8, 12.6); ax.set_ylim(-3.5, 4.6); ax.set_aspect("equal"); ax.axis("off")
band = ax.add_patch(Rectangle((0, -2.3), 0, 5.9, fc=LAVENDER, ec="none", alpha=0.35, zorder=0))
EDGES = [("x0", "h0", (0, 0)), ("x1", "h0", (0, 1)), ("x0", "h1", (1, 0)), ("x1", "h1", (1, 1)),
         ("h0", "out", 0), ("h1", "out", 1)]
edge_txt = {}
for a, bb, key in EDGES:
    (xa, ya), (xb, yb) = POS[a], POS[bb]
    ax.annotate("", xy=(xb - 0.55, yb + (ya - yb) * 0.55 / (xb - xa)), xytext=(xa + 0.55, ya - (ya - yb) * 0.55 / (xb - xa)),
                arrowprops=dict(arrowstyle="-|>", color=GREY, lw=1.3), zorder=1)
    f = 0.3 if a.startswith("x") and (a, bb) in [("x0", "h1"), ("x1", "h0")] else 0.5
    tx, ty = xa + (xb - xa) * f, ya + (yb - ya) * f
    edge_txt[(a, bb)] = ax.text(tx, ty + 0.18, "", ha="center", va="bottom", fontsize=9.5, zorder=5,
                                bbox=dict(fc="white", ec="none", alpha=.85, pad=1))
for name, (px, py) in POS.items():
    if name == "L":
        ax.add_patch(FancyBboxPatch((px - 0.6, py - 0.45), 1.2, 0.9, boxstyle="round,pad=0.05", fc=RED, ec=RED, alpha=.85, zorder=2))
        ax.text(px, py, "loss", ha="center", va="center", color="white", fontweight="bold", zorder=3); continue
    fc = BLUE if name.startswith("x") else (LAVENDER if name.startswith("h") else ORANGE)
    ax.add_patch(Circle((px, py), 0.55, fc=fc, ec=NAVY, lw=1.5, zorder=2))
    lbl = {"x0": "$x_1$", "x1": "$x_2$", "h0": "$h_1$\ntanh", "h1": "$h_2$\ntanh", "out": "$p$\nσ"}[name]
    ax.text(px, py, lbl, ha="center", va="center", fontsize=11, zorder=3, color="white" if name.startswith("x") else NAVY)
ax.annotate("", xy=(POS["L"][0] - 0.65, 1.0), xytext=(POS["out"][0] + 0.55, 1.0), arrowprops=dict(arrowstyle="-|>", color=GREY, lw=1.3))
node_txt = {n: ax.text(POS[n][0], POS[n][1] - 0.75, "", ha="center", va="top", fontsize=9.5, zorder=5, linespacing=1.3)
            for n in ["x0", "x1", "h0", "h1", "out", "L"]}
for n in ["x0", "x1"]: node_txt[n].set_position((POS[n][0] - 0.75, POS[n][1])); node_txt[n].set_ha("right"); node_txt[n].set_va("center")
bias_txt = {n: ax.text(POS[n][0], POS[n][1] + 0.62, "", ha="center", fontsize=8.5, color=GREEN) for n in ["h0", "h1", "out"]}
head = ax.text(5.4, 4.25, "", ha="center", va="center", fontsize=13, fontweight="bold")
caption = ax.text(5.4, -2.95, "", ha="center", va="center", fontsize=11.5, color=NAVY, linespacing=1.5,
                  bbox=dict(fc=LIGHT, ec=LAVENDER, boxstyle="round,pad=0.5"))

FWD, BWD = NAVY, RED
f2 = lambda v: f"{v:.2f}"
def show_weights():
    for (a, bb, key) in EDGES:
        v = W1n[key] if isinstance(key, tuple) else w2n[key]
        edge_txt[(a, bb)].set_text(f"w = {v:.2f}"); edge_txt[(a, bb)].set_color(NAVY)
    for n, v in zip(["h0", "h1", "out"], [b1n[0], b1n[1], b2n]): bias_txt[n].set_text(f"b = {v:.2f}")

# --- frames: (header, band x-range, caption, function that sets the numbers) ---
def f_start():
    show_weights()
    for t in node_txt.values(): t.set_text(""); t.set_color(NAVY)
    node_txt["x0"].set_text(f"length\n{xn[0]:.2f}"); node_txt["x1"].set_text(f"width\n{xn[1]:.2f}")
def f_hidden_z():
    for j, n in enumerate(["h0", "h1"]): node_txt[n].set_text(f"z = {z1[j]:.2f}"); node_txt[n].set_color(FWD)
def f_hidden_h():
    for j, n in enumerate(["h0", "h1"]): node_txt[n].set_text(f"z = {z1[j]:.2f}\nh = tanh(z) = {hn[j]:.2f}")
def f_out():
    node_txt["out"].set_text(f"z = {z2:.2f}\np = σ(z) = {pn:.2f}"); node_txt["out"].set_color(FWD)
def f_loss():
    node_txt["L"].set_text(f"L = −log p\n= {Ln:.2f}"); node_txt["L"].set_color(FWD)
def f_d_out():
    node_txt["out"].set_text(f"p = {pn:.2f}\nδ = p − y = {d_z2:.2f}"); node_txt["out"].set_color(BWD)
def f_g_out():
    for j, n in enumerate(["h0", "h1"]):
        edge_txt[(n, "out")].set_text(f"∂L/∂w = {d_w2[j]:.3f}"); edge_txt[(n, "out")].set_color(BWD)
    node_txt["out"].set_text(f"p = {pn:.2f}\nδ = {d_z2:.2f}\n∂L/∂b = {d_b2:.3f}")
def f_d_h():
    for j, n in enumerate(["h0", "h1"]): node_txt[n].set_text(f"h = {hn[j]:.2f}\n∂L/∂h = {d_h[j]:.3f}"); node_txt[n].set_color(BWD)
def f_d_z1():
    for j, n in enumerate(["h0", "h1"]): node_txt[n].set_text(f"h = {hn[j]:.2f}\n∂L/∂h = {d_h[j]:.3f}\nδ = {d_z1[j]:.3f}")
def f_g_hidden():
    for (a, bb, key) in EDGES[:4]:
        edge_txt[(a, bb)].set_text(f"∂L/∂w = {d_W1[key]:.3f}"); edge_txt[(a, bb)].set_color(BWD)
    for j, n in enumerate(["h0", "h1"]): node_txt[n].set_text(f"δ = {d_z1[j]:.3f}\n∂L/∂b = {d_b1[j]:.3f}")
def f_update():
    for (a, bb, key) in EDGES:
        old, new = (W1n[key], W1u[key]) if isinstance(key, tuple) else (w2n[key], w2u[key])
        edge_txt[(a, bb)].set_text(f"w: {old:.2f} → {new:.2f}"); edge_txt[(a, bb)].set_color(GREEN)
    node_txt["out"].set_text(f"p: {pn:.2f} → {pu:.2f}"); node_txt["out"].set_color(GREEN)
    node_txt["L"].set_text(f"L: {Ln:.2f} → {Lu:.2f}"); node_txt["L"].set_color(GREEN)
    for n in ["h0", "h1"]: node_txt[n].set_text("")
    for n, old, new in zip(["h0", "h1", "out"], [b1n[0], b1n[1], b2n], [b1u[0], b1u[1], b2u]): bias_txt[n].set_text(f"b: {old:.2f} → {new:.2f}")

FRAMES = [
    ("The network and one flower (a versicolor, y = 1)", None, f_start,
     "Random initial weights on the arrows. Input: the flower's standardised petal length and width."),
    ("FORWARD ① hidden layer: weighted sum", (2.6, 5.4), f_hidden_z,
     f"$z = W_1x + b_1$,   e.g.  $z_1$ = {W1n[0, 0]:.2f}·{xn[0]:.2f} + ({W1n[0, 1]:.2f})·{xn[1]:.2f} + ({b1n[0]:.2f}) = {z1[0]:.2f}"),
    ("FORWARD ② hidden layer: activation", (2.6, 5.4), f_hidden_h,
     f"$h = \\tanh(z)$:   tanh({z1[0]:.2f}) = {hn[0]:.2f},   tanh({z1[1]:.2f}) = {hn[1]:.2f}"),
    ("FORWARD ③ output layer", (6.6, 9.4), f_out,
     f"$z = w_2^\\top h + b$ = {w2n[0]:.2f}·{hn[0]:.2f} + ({w2n[1]:.2f})·{hn[1]:.2f} + ({b2n:.2f}) = {z2:.2f}     $p = \\sigma(z)$ = {pn:.2f}"),
    ("FORWARD ④ loss", (9.9, 12.1), f_loss,
     f"The flower is a versicolor, the network says p = {pn:.2f}  →  $L = -\\log p$ = {Ln:.2f}"),
    ("BACKWARD ① output layer: error signal", (6.6, 9.4), f_d_out,
     f"$\\delta_2 = \\partial L/\\partial z_2 = p - y$ = {pn:.2f} − 1 = {d_z2:.2f}     (negative: p should go UP)"),
    ("BACKWARD ② output layer: its gradients", (3.4, 9.4), f_g_out,
     f"$\\partial L/\\partial w_2 = \\delta_2 \\cdot h$:   {d_z2:.2f}·{hn[0]:.2f} = {d_w2[0]:.3f},   {d_z2:.2f}·{hn[1]:.2f} = {d_w2[1]:.3f}     "
     f"$\\partial L/\\partial b = \\delta_2$ = {d_b2:.3f}"),
    ("BACKWARD ③ hand the signal down to the hidden layer", (2.6, 9.4), f_d_h,
     f"$\\partial L/\\partial h = \\delta_2 \\cdot w_2$:   {d_z2:.2f}·{w2n[0]:.2f} = {d_h[0]:.3f},   {d_z2:.2f}·({w2n[1]:.2f}) = {d_h[1]:.3f}"),
    ("BACKWARD ④ through the activation", (2.6, 5.4), f_d_z1,
     f"$\\delta_1 = \\partial L/\\partial h \\cdot (1 - h^2)$:   {d_h[0]:.3f}·(1 − {hn[0]:.2f}²) = {d_z1[0]:.3f},   "
     f"{d_h[1]:.3f}·(1 − {hn[1]:.2f}²) = {d_z1[1]:.3f}"),
    ("BACKWARD ⑤ hidden layer: its gradients", (-0.6, 5.4), f_g_hidden,
     f"$\\partial L/\\partial W_1 = \\delta_1 \\cdot x$,  e.g.  {d_z1[0]:.3f}·{xn[0]:.2f} = {d_W1[0, 0]:.3f}      "
     f"$\\partial L/\\partial b_1 = \\delta_1$"),
    ("UPDATE: every weight takes a small step downhill", None, f_update,
     f"$w \\leftarrow w - \\eta\\,\\partial L/\\partial w$  (η = {ETA:g}).   For this flower: p {pn:.2f} → {pu:.2f}, loss {Ln:.2f} → {Lu:.2f}"),
]

def update(i):
    for k in range(i + 1): FRAMES[k][2]()       # replay all steps so far (keeps frames consistent when stepping back)
    title, rng, _, text = FRAMES[i]
    head.set_text(title); head.set_color(BWD if title.startswith("BACK") else GREEN if title.startswith("UPD") else NAVY)
    caption.set_text(text)
    if rng is None: band.set_width(0)
    else: band.set_x(rng[0]); band.set_width(rng[1] - rng[0]); band.set_facecolor(LAVENDER if title.startswith("FOR") else "#F6C9C3")
    return []

plt.tight_layout()
play(animation.FuncAnimation(fig, update, frames=len(FRAMES), interval=2200))

The same computation in code, layer by layer, compared with PyTorch's **autograd** (`loss.backward()`), which does exactly
this for any network, automatically:

In [ ]:
W1_, b1_, w2_, b2_ = net[0].weight, net[0].bias, net[2].weight[0], net[2].bias[0]

# forward, layer by layer
z1 = W1_ @ x0 + b1_;   h = torch.tanh(z1)          # hidden layer
z2 = w2_ @ h + b2_;    p = torch.sigmoid(z2)       # output layer
loss = -(y0 * torch.log(p) + (1 - y0) * torch.log(1 - p))

# backward by hand, layer by layer: output → hidden
with torch.no_grad():
    d_z2 = p - y0                                  # output layer: error signal
    d_w2, d_b2 = d_z2 * h, d_z2                    #   → its gradients
    d_h = d_z2 * w2_                               # hand the signal down
    d_z1 = d_h * (1 - h**2)                        # through tanh
    d_W1, d_b1 = torch.outer(d_z1, x0), d_z1       #   → gradients of the hidden layer

loss.backward()                                    # autograd
for name, mine, auto in [("W1", d_W1, net[0].weight.grad), ("b1", d_b1, net[0].bias.grad),
                         ("w2", d_w2, net[2].weight.grad[0]), ("b2", d_b2, net[2].bias.grad[0])]:
    fmt = lambda t: np.array2string(t.numpy(), precision=4, suppress_small=True).replace("\n", "")
    print(f"{name}   by hand: {fmt(mine):<32} autograd: {fmt(auto)}")
net.zero_grad()

## 3.4 · The training loop
Every training script you will ever see has this skeleton:

```python
for epoch in range(num_epochs):          # one epoch = one pass over the whole dataset
    for x, y in train_loader:            # one mini-batch
        y_hat = model(x)                 # 1. forward pass
        loss  = loss_fn(y_hat, y)        # 2. loss
        optimizer.zero_grad()            #    (reset old gradients)
        loss.backward()                  # 3. backpropagation
        optimizer.step()                 # 4. update: w ← w − η ∂L/∂w
```

Instead of computing the loss on all flowers at once, we feed **mini-batches** of 5 random flowers: every step is cheap,
and after 30 steps the network has seen all 150 flowers once (one **epoch**). Let's train our 2 → 2 → 1 network on **Task B**,
starting from the same random weights as in the backpropagation animation:

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(SEED)
model = nn.Sequential(nn.Linear(2, 2), nn.Tanh(), nn.Linear(2, 1))      # 2 → 2 → 1, same start as above
X_t, y_t = torch.tensor(X_all), torch.tensor(y_all)
loader = DataLoader(TensorDataset(X_t, y_t, torch.arange(150)), batch_size=5, shuffle=True,
                    generator=torch.Generator().manual_seed(0))         # mini-batches of 5 flowers (+ their index)
loss_fn = nn.BCEWithLogitsLoss()                                        # sigmoid + binary cross-entropy in one
optimizer = torch.optim.SGD(model.parameters(), lr=1.0)                 # plain gradient descent

train_log = []                                                          # remember every step for the animation
for epoch in range(6):
    for xb, yb, idx in loader:
        logits = model(xb).squeeze(1)       # 1. forward pass
        loss = loss_fn(logits, yb)          # 2. loss
        optimizer.zero_grad()
        loss.backward()                     # 3. backpropagation
        train_log.append(dict(idx=idx.numpy(), loss=loss.item(),
                              params=[q.detach().numpy().copy() for q in model.parameters()],
                              grads=[q.grad.numpy().copy() for q in model.parameters()]))
        optimizer.step()                    # 4. update
train_log.append(dict(params=[q.detach().numpy().copy() for q in model.parameters()]))   # final weights

with torch.no_grad():
    acc = ((model(X_t).squeeze(1) > 0).float() == y_t).float().mean()
print(f"{len(train_log) - 1} steps ({epoch + 1} epochs × {len(loader)} mini-batches)  →  accuracy on all 150 flowers: {acc:.0%}")

### 🎞️ Watch the training loop
The first **three steps in slow motion**, line by line, then **fast forward** through all six epochs.

* **left:** the training loop (the line being executed is highlighted) and the loss on all 150 flowers
* **middle:** the network's answer everywhere; the 5 flowers of the current mini-batch are enlarged and numbered.
  During the update the old boundary is shown dashed.
* **right:** the mini-batch (prediction vs. truth and loss per flower) and the 9 gradients, filled in by
  backpropagation **layer by layer**: output layer first, then the hidden layer.

In [ ]:
# @title 🎞️ Animation: the training loop, step by step { display-mode: "form" }
def forward_np(params, X):
    W1, b1, W2, b2 = params
    H = np.tanh(X @ W1.T + b1)
    return 1 / (1 + np.exp(-(H @ W2[0] + b2[0])))

def bce_np(p, y):
    p = np.clip(p, 1e-7, 1 - 1e-7); return -(y * np.log(p) + (1 - y) * np.log(1 - p))

STEPS = len(train_log) - 1
full_loss = np.array([bce_np(forward_np(train_log[s]["params"], X_all), y_all).mean() for s in range(STEPS + 1)])
full_acc = np.array([((forward_np(train_log[s]["params"], X_all) > .5) == y_all).mean() for s in range(STEPS + 1)])
grad_lim = np.percentile([np.abs(np.concatenate([g.ravel() for g in train_log[s]["grads"]])).max() for s in range(STEPS)], 90) * 1.1

CODE = ["for epoch in range(6):", "    for xb, yb in loader:", "        logits = model(xb)", "        loss = loss_fn(logits, yb)",
        "        optimizer.zero_grad()", "        loss.backward()", "        optimizer.step()"]
PHASES = [("batch", [1], "Pick a mini-batch: 5 random flowers"),
          ("forward", [2], "Forward pass: the network guesses p(versicolor)\nfor each of the 5 flowers"),
          ("loss", [3], "Loss: compare each guess with the truth,\nthen average over the mini-batch"),
          ("zero", [4], "Reset the gradients of the previous step to 0"),
          ("back2", [5], "Backpropagation, part 1:\ngradients of the OUTPUT layer"),
          ("back1", [5], "Backpropagation, part 2: the signal is handed\ndown → gradients of the HIDDEN layer"),
          ("step", [6], "Update: every weight takes a step downhill\nw ← w − η·∂L/∂w   (dashed = old boundary)")]
frames = [(s, ph) for s in range(3) for ph in PHASES] + [(s, None) for s in range(4, STEPS, 4)] + [(STEPS, None)]

fig = plt.figure(figsize=(16, 7.4), dpi=80)
gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.25, 1.0], height_ratios=[1, 1], wspace=0.28, hspace=0.42)
ax_code, ax_curve = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[1, 0])
ax_data = fig.add_subplot(gs[:, 1]); ax_batch, ax_grad = fig.add_subplot(gs[0, 2]), fig.add_subplot(gs[1, 2])

# code panel
ax_code.axis("off"); ax_code.set_xlim(0, 1); ax_code.set_ylim(0, 1)
hl = ax_code.add_patch(Rectangle((0, 0), 1, 0.105, fc=ORANGE, alpha=.3, ec="none"))
code_y = [0.92 - 0.11 * i for i in range(len(CODE))]
for yy, line in zip(code_y, CODE): ax_code.text(0.02, yy, line, family="monospace", fontsize=10.5, va="center", color=NAVY)
phase_txt = ax_code.text(0.02, 0.06, "", fontsize=10.5, va="center", color=RED, fontweight="bold")
ax_code.set_title("The training loop", loc="left")

# loss curve
ax_curve.plot(full_loss, color=NAVY, lw=.8, alpha=.2)
curve, = ax_curve.plot([], [], color=NAVY, lw=2.2, label="loss on all 150 flowers")
batch_dots, = ax_curve.plot([], [], ".", color=ORANGE, ms=3, alpha=.5, label="loss of each mini-batch")
cursor = ax_curve.axvline(0, color=GREY, lw=.8, ls=":")
for e in range(1, STEPS // 30): ax_curve.axvline(30 * e, color=LAVENDER, lw=.6)
ax_curve.set(xlabel="step   (30 steps = 1 epoch)", ylabel="loss", xlim=(0, STEPS), ylim=(0, max(full_loss.max(), 1.0) * 1.05))
ax_curve.legend(fontsize=8, loc="upper right")

# data panel
plot_flowers(ax_data, s=22, legend=True)
bg = ax_data.imshow(np.zeros(G1.shape), extent=(*EXT, *EXT), origin="lower", cmap=P_CMAP, vmin=0, vmax=1, alpha=.55, zorder=0)
batch_sc = ax_data.scatter([], [], s=190, facecolors="none", edgecolors="black", lw=2, zorder=5)
num_txt = [ax_data.text(0, 0, str(k + 1), fontsize=10, fontweight="bold", zorder=6, ha="center", va="center",
                        bbox=dict(boxstyle="circle,pad=0.15", fc="white", ec="black", lw=1)) for k in range(5)]
lines = {"new": None, "old": None}

# mini-batch panel: predicted p (bar) vs. truth (diamond)
rows = np.arange(5)[::-1]
pbars = ax_batch.barh(rows, np.zeros(5), height=.55, color=GREY)
truth = ax_batch.scatter(np.zeros(5), rows, marker="D", s=60, c=NAVY, zorder=4, clip_on=False)
after = ax_batch.scatter(np.full(5, np.nan), rows, marker="|", s=380, c=GREEN, lw=3, zorder=5, clip_on=False)
loss_txt = [ax_batch.text(1.03, r, "", va="center", fontsize=9, color=RED) for r in rows]
ax_batch.set(xlim=(0, 1), ylim=(-0.7, 4.7), yticks=rows, xlabel="p(versicolor):  bar = prediction,  ◆ = truth,  ┃ = after the update")
ax_batch.axvline(.5, color=GREY, lw=.8, ls=":")
batch_title = ax_batch.set_title("The mini-batch", loc="left")

# gradient panel: 9 bars, hidden layer (left) and output layer (right)
names = ["$W_1$[1,1]", "$W_1$[1,2]", "$W_1$[2,1]", "$W_1$[2,2]", "$b_1$[1]", "$b_1$[2]", "$w_2$[1]", "$w_2$[2]", "$b_2$"]
xs = np.array([0, 1, 2, 3, 4, 5, 6.8, 7.8, 8.8])
gbars = ax_grad.bar(xs, np.zeros(9), color=[BLUE] * 6 + [ORANGE] * 3, width=.7)
gtxt = [ax_grad.text(x, 0, "", ha="center", fontsize=7, color=NAVY) for x in xs]
ax_grad.axhline(0, color=GREY, lw=.8); ax_grad.set_xticks(xs); ax_grad.set_xticklabels(names, rotation=60, fontsize=8)
ax_grad.set_ylim(-grad_lim, grad_lim); ax_grad.set_ylabel("∂L/∂w")
ax_grad.text(2.5, grad_lim * 0.92, "hidden layer", ha="center", color=BLUE, fontsize=9, fontweight="bold")
ax_grad.text(7.8, grad_lim * 0.92, "output layer", ha="center", color=ORANGE, fontsize=9, fontweight="bold")
ax_grad.set_title("Gradients (filled in by backprop)", loc="left")
fig.suptitle("", fontsize=13, fontweight="bold", color=NAVY); sup = fig._suptitle

def set_boundary(key, params, **kw):
    if lines[key] is not None: lines[key].remove()
    lines[key] = contour_at(ax_data, forward_np(params, GRID), 0.5, zorder=4, **kw)

def update(i):
    s, ph = frames[i]; log = train_log[s]; params = log["params"]
    name = ph[0] if ph else "ff"
    order = ["batch", "forward", "loss", "zero", "back2", "back1", "step"]
    k = order.index(name) if ph else 99                       # how far into this step we are
    # code highlight + text
    hl_lines = ph[1] if ph else list(range(1, 7))
    hl.set_y(code_y[hl_lines[-1]] - 0.055); hl.set_height(0.11 * len(hl_lines)); hl.set_y(code_y[hl_lines[-1]] - 0.055)
    phase_txt.set_text(ph[2] if ph else "▶▶ fast forward: one frame = 4 steps")
    epoch = min(s // 30 + 1, STEPS // 30)
    sup.set_text(f"epoch {epoch} · step {s}" + ("   (slow motion)" if ph else ""))
    # data panel
    shown = train_log[s + 1]["params"] if (ph and k >= 6) else params
    bg.set_data(forward_np(shown, GRID).reshape(G1.shape))
    set_boundary("new", shown, colors=NAVY, linewidths=2.4)
    if ph and k >= 6: set_boundary("old", params, colors=NAVY, linewidths=1.6, linestyles="--")
    elif lines["old"] is not None: lines["old"].remove(); lines["old"] = None
    ax_data.set_title(f"accuracy on all 150 flowers: {full_acc[s + 1 if (ph and k >= 6) else s]:.0%}")
    # curve
    upto = min(s + 1 if (ph and k >= 6) or not ph else s, STEPS)
    curve.set_data(np.arange(upto + 1), full_loss[:upto + 1]); cursor.set_xdata([s, s])
    batch_dots.set_data(np.arange(min(upto, STEPS)), [train_log[j]["loss"] for j in range(min(upto, STEPS))])
    if s >= STEPS:                                           # last frame: final weights, no mini-batch
        batch_sc.set_offsets(np.empty((0, 2))); [t.set_visible(False) for t in num_txt]
        for b in pbars: b.set_width(0)
        truth.set_offsets(np.c_[np.full(5, np.nan), rows]); [t.set_text("") for t in loss_txt]
        after.set_offsets(np.c_[np.full(5, np.nan), rows]); ax_batch.set_yticklabels([""] * 5); batch_title.set_text("The mini-batch")
        for b, t in zip(gbars, gtxt): b.set_height(0); t.set_text("")
        phase_txt.set_text("Done! The network found the band by itself."); return []
    idx = log["idx"]; Xb, yb = X_all[idx], y_all[idx]
    batch_sc.set_offsets(Xb)
    for t, (px, py) in zip(num_txt, Xb): t.set_visible(True); t.set_position((px + 0.17, py + 0.17))
    # mini-batch panel
    pb = forward_np(params, Xb); lb = bce_np(pb, yb)
    truth.set_offsets(np.c_[yb, rows])
    for b, pv, sp in zip(pbars, pb, species[idx]):
        b.set_width(pv if k >= 1 else 0); b.set_color(SP_COLORS[sp])
    for t, lv in zip(loss_txt, lb): t.set_text(f"L = {lv:.2f}" if k >= 2 else "")
    ax_batch.set_yticklabels([f"{j + 1} · {SPECIES[sp]}" for j, sp in enumerate(species[idx])])
    batch_title.set_text(f"The mini-batch   (mean loss {lb.mean():.2f})" if k >= 2 else "The mini-batch")
    if ph and k >= 6: after.set_offsets(np.c_[forward_np(train_log[s + 1]["params"], Xb), rows])
    else: after.set_offsets(np.c_[np.full(5, np.nan), rows])
    # gradient panel
    g = np.concatenate([q.ravel() for q in log["grads"]])
    show = np.zeros(9)
    if k <= 2 and s > 0: show = np.concatenate([q.ravel() for q in train_log[s - 1]["grads"]])   # still there from the last step
    if k >= 4: show[6:] = g[6:]
    if k >= 5: show[:6] = g[:6]
    for b, t, v in zip(gbars, gtxt, show):
        b.set_height(np.clip(v, -grad_lim, grad_lim))
        t.set_text(f"{v:.2f}" if v != 0 else ""); t.set_y(np.clip(v, -grad_lim, grad_lim) + (0.02 if v >= 0 else -0.06) * grad_lim * 2)
    return []

play(animation.FuncAnimation(fig, update, frames=len(frames), interval=650))

Notice the **plateau** in the loss curve: for about two epochs the network answers "not a versicolor" for *every* flower.
That is already 67 % correct, because two thirds of the flowers are not versicolors. Then it discovers the band and the loss
drops. Also note that single mini-batches are noisy (orange dots): a step can make things worse for some flowers. On average
the steps still go downhill.

### What did the network learn?
Let's look inside the trained network, exactly like we did with the hand-picked weights in Section 2:

In [ ]:
W1_t, b1_t, W2_t, b2_t = [q.detach().numpy().astype(float).round(2) for q in model.parameters()]
fig, axes = plt.subplots(1, 3, figsize=(12, 4.4))
network_maps(axes, W1_t, b1_t, W2_t[0], float(b2_t[0]))
fig.suptitle("The trained 2 → 2 → 1 network (weights rounded to 2 decimals)", color=NAVY, fontweight="bold")
plt.tight_layout(rect=(0, 0, 1, 0.92)); plt.show()

Nobody told the network about "bigger than a setosa" or "bigger than a versicolor". Starting from random weights, gradient
descent found **two lines and a way to combine them**, just like our hand-built solution, only by minimising the loss.

### The learning rate matters
The same training with three learning rates. Too small: after six epochs the network has barely moved. Too large: every
step overshoots, the loss jumps around and the network never settles.

In [ ]:
def train_with_lr(lr, epochs=6):
    torch.manual_seed(SEED)
    net = nn.Sequential(nn.Linear(2, 2), nn.Tanh(), nn.Linear(2, 1))
    opt = torch.optim.SGD(net.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=5, shuffle=True, generator=torch.Generator().manual_seed(0))
    losses = []
    for epoch in range(epochs):
        for xb, yb in loader:
            loss = loss_fn(net(xb).squeeze(1), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad(): losses.append(loss_fn(net(X_t).squeeze(1), y_t).item())   # loss on all flowers
    return losses

plt.figure(figsize=(7, 3.6))
for lr_, c, note in [(0.1, GREEN, "too small"), (1.0, NAVY, "good"), (10.0, RED, "too large")]:
    plt.plot(train_with_lr(lr_), color=c, lw=2, label=f"η = {lr_:g}  ({note})")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss on all 150 flowers"); plt.title("Effect of the learning rate")
plt.legend(); plt.show()

---
# Summary
1. A **neuron** computes a weighted sum plus a bias, followed by an activation: $y = \sigma(\mathbf{w}^\top\mathbf{x} + b)$.
   With two inputs it draws **one straight line**.
2. Problems like Task B need several lines combined: **hidden layers**. Without **non-linear activations** the layers collapse
   into a single linear map, and we are back to one line.
3. The **loss** measures how wrong the network is; **gradient descent** walks downhill on the loss landscape;
   the **learning rate** sets the step size.
4. **Backpropagation** computes all gradients with the chain rule, from the output layer back to the input, one layer at a time.
5. The **training loop** repeats: mini-batch → forward pass → loss → zero_grad → backward → step.

**Next →** [01 · Computer Vision & an MLP on MNIST](https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/01_computer_vision_and_mlp_mnist.ipynb): what happens when the input is an image?